In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

In [ ]:
# Load dataset - UPDATE THIS PATH to your local file location
df = pd.read_csv("train (1).csv")  # Change to your actual path

In [ ]:
# Fix missing values (avoid inplace chained assignment warnings)
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

numeric_cols = ['LoanAmount', 'Loan_Amount_Term']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

# Verify no missing values remain
print("Missing values after imputation:")
print(df.isnull().sum())

In [ ]:
# Define RAW feature names (what users will input)
raw_categorical_features = [
    'Gender', 'Married', 'Dependents',
    'Education', 'Self_Employed', 'Property_Area'
]

raw_numeric_features = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

print("Raw categorical features:", raw_categorical_features)
print("Raw numeric features:", raw_numeric_features)

In [ ]:
# Create a custom transformer for feature engineering (log transforms + total income)
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer to create engineered features:
    - Log transforms of ApplicantIncome, LoanAmount, Loan_Amount_Term
    - Total_Income and its log
    """
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        # Log transforms
        X_copy['ApplicantIncomeLog'] = np.log(X_copy['ApplicantIncome'] + 1)
        X_copy['LoanAmountLog'] = np.log(X_copy['LoanAmount'] + 1)
        X_copy['Loan_Amount_Term_Log'] = np.log(X_copy['Loan_Amount_Term'] + 1)
        # Total income
        X_copy['Total_Income'] = X_copy['ApplicantIncome'] + X_copy['CoapplicantIncome']
        X_copy['Total_Income_Log'] = np.log(X_copy['Total_Income'] + 1)
        return X_copy

# Create preprocessing pipeline that includes feature engineering
preprocessor = Pipeline([
    ('feature_engineering', FeatureEngineer()),
    ('column_transformer', ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), raw_categorical_features),
            ('num', 'passthrough', [
                'ApplicantIncomeLog', 'LoanAmountLog',
                'Loan_Amount_Term_Log', 'Total_Income_Log', 'Credit_History'
            ])
        ]
    ))
])

In [ ]:
# Encode target variable
y = df['Loan_Status'].map({'Y': 1, 'N': 0})

# Use ONLY raw features for X (the pipeline will engineer new features internally)
X = df[raw_categorical_features + raw_numeric_features]

print("X columns:", X.columns.tolist())
print("y value counts:")
print(y.value_counts())

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
# Create full pipeline with preprocessing and XGBoost model
from xgboost import XGBClassifier

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])

# Train the pipeline
print("Training pipeline...")
pipeline.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Evaluate
accuracy = pipeline.score(X_test, y_test)
print(f"Accuracy: {accuracy:.4f}")

# Optional: more detailed metrics
from sklearn.metrics import classification_report, confusion_matrix
y_pred = pipeline.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Save the complete pipeline
output_file = "loan_model_pipeline.pkl"
joblib.dump(pipeline, output_file)
print(f"Pipeline saved to: {output_file}")

In [ ]:
# ============================================
# IMPORTANT: Verify the pipeline accepts RAW inputs
# ============================================

# Create a test input with RAW values only (what a user would provide)
test_input_raw = pd.DataFrame([{
    'Gender': 'Male',
    'Married': 'No',
    'Dependents': '0',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'Property_Area': 'Urban',
    'ApplicantIncome': 6000,    # Raw value
    'CoapplicantIncome': 0,     # Raw value
    'LoanAmount': 150,          # Raw value
    'Loan_Amount_Term': 360,    # Raw value
    'Credit_History': 1
}])

print("Test input (raw features):")
print(test_input_raw)

# Predict using RAW input - the pipeline will do all transformations internally
prediction = pipeline.predict(test_input_raw)
print(f"\nPrediction: {prediction}")
print("Interpretation: 1 = Loan Approved, 0 = Loan Rejected")

# Check probabilities if available
if hasattr(pipeline, 'predict_proba'):
    proba = pipeline.predict_proba(test_input_raw)
    print(f"Probabilities: [Rejection: {proba[0][0]:.4f}, Approval: {proba[0][1]:.4f}]")

In [ ]:
# Verify saved pipeline
print("Verifying saved pipeline...")
loaded_pipeline = joblib.load(output_file)
verification_pred = loaded_pipeline.predict(test_input_raw)
print(f"Loaded pipeline prediction: {verification_pred}")
print(f"Match: {prediction[0] == verification_pred[0]}")
print("✓ Pipeline verified!")

In [ ]:
# Check what feature names the pipeline expects (should be raw features)
print("\n=== Feature Schema Information ===")
print("Expected raw input features (what users should provide):")
print("Categorical:", raw_categorical_features)
print("Numeric:", raw_numeric_features)

# The ColumnTransformer inside the pipeline should have feature_names_in_
col_transformer = pipeline.named_steps['preprocessing'].named_steps['column_transformer']
if hasattr(col_transformer, 'feature_names_in_'):
    print("\nFeatures recognized by ColumnTransformer:")
    print(col_transformer.feature_names_in_.tolist())
else:
    print("\nWarning: ColumnTransformer does not have feature_names_in_ attribute")
    print("Ensure you fit the pipeline on a DataFrame (not numpy array) to preserve feature names.")

## 📤 Next Steps

1. **Update the file path** in cell 2 to point to your actual CSV location
2. **Run all cells sequentially** (Kernel → Restart & Run All)
3. **Verify** the test prediction in cell 8 works with RAW values (no manual log transforms!)
4. **Upload** `loan_model_pipeline.pkl` to your XAI platform
5. **Test** from the frontend - it should now accept raw values like:
   - Gender: `Male` / `Female`
   - ApplicantIncome: `6000`
   - LoanAmount: `150`
   etc.

The pipeline will automatically:
1. Create log transforms
2. One-hot encode categoricals
3. Make prediction

✅ No manual preprocessing needed by the user!